# 2.6 Annotated Assignment and Type Hints

## Overview

Annotated assignment belongs to Python's **language semantics**; type hints are a convention and a formal **static type system** layered on that syntax. The distinction is critical. An annotation can participate in symbol classification and introspection without producing a runtime value, enforcing a constraint, or guaranteeing that financial data is valid.

This notebook studies assignment annotations, annotation evaluation across Python releases, generic types, `TypedDict`, structural protocols, runtime narrowing, overloaded interfaces, metadata, and annotation-driven tooling. It culminates in a **type-safe multi-currency portfolio valuation and risk-contract pipeline** that separates static contracts from defensive runtime validation.

### Learning Objectives

1. Interpret the grammar and runtime effects of annotated assignment in module, class, and function scopes.
2. Differentiate annotation-only bindings, ordinary assignments, and local variable classification.
3. Explain annotation introspection and the differences between Python 3.10–3.13 and Python 3.14+.
4. Contrast `Any`, `object`, unions, `Literal`, and `None` in static analyses.
5. Apply `Sequence`, `Mapping`, mutable collections, `tuple`, and safe generic interfaces.
6. Distinguish nominal units using `NewType` and readable type aliases without mistaking them for runtime validation.
7. Model shaped records using `TypedDict` and inspect required/optional keys.
8. Use `Annotated` metadata and correctly interpret `get_type_hints(include_extras=True)`.
9. Implement reusable generic algorithms, `TypeVar`, and `ParamSpec` signature-preserving decorators.
10. Specify structural contracts using `Protocol` and understand runtime protocol-check limitations.
11. Apply `@overload`, `TypeGuard`, `cast`, `Final`, `ClassVar`, and `InitVar` responsibly.
12. Inspect annotations and name-binding structure using runtime introspection and AST.
13. Compare optional strict static checking with financial runtime input validation.
14. Build a Decimal-based multi-currency valuation and risk engine with immutable snapshots.
15. Verify numerical invariants, precision, input isolation, type assumptions, and failure paths.

### Version Compatibility

The executable core targets **Python 3.10+**, including Python 3.13. Modern features introduced in later releases (3.11 `NotRequired`, 3.12 `type` aliases and inline type parameters, 3.13 `TypeIs`, and 3.14 default lazy annotation evaluation) are explicitly discussed or demonstrated through version-gated examples. **Do not conflate version-specific runtime annotation semantics.**

### Notebook Organization

| Section | Advanced Topic |
|---|---|
| 2.6.1 | Annotated assignment grammar, storage, and local bindings |
| 2.6.2 | Evaluation strategies, postponed annotations, and version gates |
| 2.6.3 | Static vs runtime contracts and financial validation |
| 2.6.4 | `Any`, `object`, unions, `Literal`, and safe narrowing |
| 2.6.5 | Parameterized collections and variance-sensitive APIs |
| 2.6.6 | Type aliases, `NewType`, and dimensional correctness |
| 2.6.7 | `TypedDict` record schemas and optional fields |
| 2.6.8 | `Annotated` metadata and type-hint introspection |
| 2.6.9 | `TypeVar`, `Generic`, `ParamSpec`, and reusable functions |
| 2.6.10 | Structural subtyping and runtime-checkable protocols |
| 2.6.11 | Overloads, type guards, and casts |
| 2.6.12 | `Final`, `ClassVar`, `InitVar`, and dataclass contracts |
| 2.6.13 | AST auditing and static-checker workflows |
| 2.6.14 | Practical Application — Typed FX Portfolio Valuation & Risk Gate |
| 2.6.15 | Automated tests and numerical invariants |
| 2.6.16 | Final Analytical Interpretation |
| — | Advanced Practice Exercises / Advanced Challenge |

**Primary sources:** [Python Language Reference — annotated assignments](https://docs.python.org/3/reference/simple_stmts.html#annotated-assignment-statements), [Python `typing` reference](https://docs.python.org/3/library/typing.html), [Python annotation best practices](https://docs.python.org/3/howto/annotations.html), [Static typing specification](https://typing.python.org/en/latest/spec/), [Type narrowing guide](https://typing.python.org/en/latest/guides/type_narrowing.html).

**Scope boundary:** The syntax and binding mechanics of annotations belong here. Detailed descriptor assignment was covered in 2.2; packaging, dependency management, and full static-analysis CI pipelines will be developed in later engineering chapters.

## 2.6.1 Annotated Assignment Grammar and Binding

### Theoretical Foundations

### Formal Definition

An annotated assignment combines an **annotation target**, a type expression, and an optional assigned value:

```python
exposure_usd: float = 125_000.0
uninitialized_exposure: float
```

The second form **does not assign a value**. At module/class scope, annotations can be recorded in the corresponding `__annotations__` mapping. At function scope, annotating a simple name determines its **local-name classification**; an annotation alone neither initializes the local variable nor creates a conventional local `__annotations__` entry.

### Important Target Distinctions

- `name: T = expr`: annotated simple name and ordinary assignment effect.
- `name: T`: annotation only; no value binding.
- `obj.attr: T = expr`: performs attribute assignment; it does **not** add `attr` to the enclosing module's `__annotations__` mapping.
- `items[0]: T = expr`: performs subscription assignment; the target can involve custom `__setitem__` behavior.
- `obj.attr: T` without a value has **no target evaluation**; it does not read the attribute or invoke a setter.
- Chained assignment does not accept this annotation form: `a: int = b = 1` is invalid.

### Symbol Classification

Within a function, annotation of a simple name makes that name local for the whole code block. Reading before assignment may raise `UnboundLocalError` even when a global binding with the same spelling exists.

### Annotation Evaluation Warning

The *evaluation* of annotation expressions differs by Python version and by use of `from __future__ import annotations`; this will be treated explicitly in 2.6.2.

In [1]:
import ast
import sys

ann_value: int = 125
ann_only: float
assert ann_value == 125
assert "ann_value" in globals()["__annotations__"]
assert "ann_only" in globals()["__annotations__"]
assert "ann_only" not in globals()  # fresh notebook execution

class AnnotationAccount:
    balance: float
    currency: str = "USD"

assert "balance" in AnnotationAccount.__annotations__
assert not hasattr(AnnotationAccount, "balance")
assert AnnotationAccount.currency == "USD"

# The annotated attribute expression is not evaluated without an RHS.
class ExplosiveAttribute:
    @property
    def loss(self) -> float:
        raise RuntimeError("unexpected property evaluation")

obj_ann = ExplosiveAttribute()
obj_ann.loss: float  # no RHS: no attribute lookup or setter

# Compile-time local classification, despite the global binding.
ann_shadow_global = "global object"
def demonstrate_local_annotation() -> str:
    ann_shadow_global: str
    return ann_shadow_global

try:
    demonstrate_local_annotation()
except UnboundLocalError:
    print("Local annotation caused uninitialized local-name lookup")

structure = ast.parse("capital: float = 10.0\nother: int")
assert [type(x).__name__ for x in structure.body] == ["AnnAssign", "AnnAssign"]
print("Module annotations:", {k: str(v) for k, v in globals()["__annotations__"].items() if k in ("ann_value", "ann_only")})
print("Python version:", sys.version.split()[0])

Local annotation caused uninitialized local-name lookup
Module annotations: {'ann_value': "<class 'int'>", 'ann_only': "<class 'float'>"}
Python version: 3.13.5


### **Analytical Interpretation**

**1. Annotation-only is not initialization.** `ann_only` is recorded as annotated but has no runtime binding; attempting to read it would raise a name-resolution error.

**2. Class annotations are distinct from attributes.** `AnnotationAccount.balance` is documented through the class annotation mapping but has no assigned class value.

**3. No RHS, no complex-target evaluation.** The `obj_ann.loss: float` statement does not invoke the property getter. It therefore cannot be treated like `obj_ann.loss = ...`.

**4. Function-level annotation affects compilation.** The local-name classification triggers `UnboundLocalError` before the function can use the global name.

**5. AST classification.** The `AnnAssign` node captures target, annotation expression, and optional assigned value. Static tools must treat it differently from `Assign`.

**6. Financial consequence.** Annotating a portfolio amount does not allocate cash, validate its units, or ensure it is initialized. All three require additional runtime guarantees.

## 2.6.2 Annotation Evaluation, Postponement, and Version Compatibility

### Theoretical Foundations

### Runtime Annotation Models

For Python **3.10–3.13**, module/class annotation expressions are generally evaluated eagerly under ordinary semantics (subject to the annotated-target rules). `from __future__ import annotations` changes the storage/evaluation behavior by postponing resolution and storing stringized expressions in these versions.

For Python **3.14+**, annotation expressions are **lazily evaluated by default** (PEP 649/749 implementation), with a new `annotationlib` introspection API. The future import remains supported but has distinct behavior; do not describe 3.14 as simply the 3.13 model.

### Introspection APIs

- `__annotations__`: the raw annotation view, subject to version-specific semantics.
- `inspect.get_annotations()` (available in Python 3.10+) supports annotation introspection and optional evaluation of string expressions.
- `typing.get_type_hints()` resolves annotation references and transforms some typing constructs; it can execute code contained in annotations/forward references. **Never evaluate untrusted annotations.**
- In Python 3.14+, `annotationlib` supports value, forward-reference, and string formats; it is not available in the 3.10 standard library.

### `TYPE_CHECKING`

The flag `typing.TYPE_CHECKING` is `False` during ordinary execution and is interpreted as `True` by type checkers. It can avoid runtime-only imports of static dependencies, but unresolved names may still prevent some runtime annotation evaluation unless introspection is planned carefully.

### Portability Principle

Write executable examples using syntax available in Python 3.10. Demonstrate later syntax using **version-gated compilation**, not unguarded syntax that fails to parse on older interpreters.

In [2]:
import inspect
import sys
import typing

annotation_source = """
from __future__ import annotations

def forecast(volatility: "RiskScale") -> "RiskScale":
    return volatility
"""
source_ns = {}
exec(compile(annotation_source, "<our_controlled_demo>", "exec"), source_ns)
forecast = source_ns["forecast"]
raw_forecast_annotations = inspect.get_annotations(forecast, eval_str=False)
assert all(isinstance(x, str) for x in raw_forecast_annotations.values())
print("Postponed source annotations:", raw_forecast_annotations)

# All ordinary examples in this notebook use Python 3.10-compatible syntax.
if sys.version_info >= (3, 14):
    import annotationlib
    print("Annotationlib supported:", hasattr(annotationlib, "get_annotations"))
else:
    print("Annotationlib is Python 3.14+; running portable path")

if sys.version_info >= (3, 12):
    alias_ns = {}
    exec(compile("type CurrencyCode = str", "<modern_type_alias>", "exec"), alias_ns)
    print("PEP 695 alias runtime type:", type(alias_ns["CurrencyCode"]).__name__)
else:
    print("PEP 695 `type` alias syntax requires Python 3.12+")

assert typing.TYPE_CHECKING is False

Postponed source annotations: {'volatility': "'RiskScale'", 'return': "'RiskScale'"}
Annotationlib is Python 3.14+; running portable path
PEP 695 alias runtime type: TypeAliasType


### **Analytical Interpretation**

**1. Postponed annotations are not runtime checks.** The future import affects annotation representation; it does not validate inputs to `forecast`.

**2. Controlled code generation.** `exec()` is used only for trusted, fixed instructional source in an isolated namespace. Never adapt this pattern to untrusted text.

**3. Version-gated new syntax.** Python 3.12+ `type` statements are compiled only on interpreters supporting the syntax.

**4. Python 3.14 changes matter.** Default deferred annotation evaluation and `annotationlib` cannot be inferred from behavior observed on Python 3.10–3.13.

**5. Introspection security.** Resolving annotation expressions can execute code. Runtime annotations supplied from unknown sources must not be blindly evaluated.

**6. Research implication.** Cross-version reproducibility requires recording Python version and annotation-introspection conventions in analytical project metadata.

## 2.6.3 Static Type Contracts Versus Runtime Financial Validation

### Theoretical Foundations

### Type Hints Do Not Enforce Values

`price: Decimal` is a declaration of *intent* to type checkers and human readers. Normal Python does not automatically prevent assignment of a `str` to that name. Financial invariants must be enforced independently.

### Three Layers of Correctness

1. **Static**: a checker proves/flags relationships in annotated source under its model.
2. **Runtime structural**: an input has the required representation (e.g. `str`, `Decimal`, `Mapping`).
3. **Domain**: price > 0, FX quote finite, currency supported, quantity nonzero, and accounting units consistent.

`bool` subclasses `int`; an `isinstance(True, int)` test is insufficient for quantity validation. Likewise, converting floats directly with `Decimal(float)` preserves their binary approximation rather than decimal business intent.

### Precision and Rounding

Prefer `Decimal` constructed from **decimal text** for money or cost-basis demonstrations. Define an explicit quantization/rounding policy at actual reporting or settlement boundaries; premature rounding can bias valuations.

In [3]:
from decimal import Decimal, InvalidOperation, ROUND_HALF_EVEN

stated_exposure: Decimal = "incorrect but not automatically rejected"
assert isinstance(stated_exposure, str)

def validated_decimal(value: object, *, strictly_positive: bool = False) -> Decimal:
    """Accept Decimal or decimal string and enforce financial value constraints."""
    if not isinstance(value, (str, Decimal)):
        raise TypeError("Financial Decimal must be Decimal or decimal text")
    try:
        amount = Decimal(value)
    except InvalidOperation as exc:
        raise ValueError("Invalid decimal amount") from exc
    if not amount.is_finite():
        raise ValueError("Nonfinite decimal amount")
    if strictly_positive and amount <= 0:
        raise ValueError("A strictly positive amount is required")
    return amount

assert validated_decimal("0.10") + validated_decimal("0.20") == Decimal("0.30")
assert Decimal(0.1) != Decimal("0.1")
print("Explicit decimal arithmetic:", Decimal("0.10") + Decimal("0.20"))

for value in (True, 0.1, "NaN", "Infinity", "-1"):
    try:
        validated_decimal(value, strictly_positive=True)
    except (TypeError, ValueError):
        print("Rejected:", repr(value))

amount = Decimal("102.345")
print("Reporting quantization:", amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_EVEN))

Explicit decimal arithmetic: 0.30
Rejected: True
Rejected: 0.1
Rejected: 'NaN'
Rejected: 'Infinity'
Rejected: '-1'
Reporting quantization: 102.34


### **Analytical Interpretation**

**1. Static promise is not runtime enforcement.** A misleading value is assigned despite the variable's `Decimal` annotation. Static checkers are useful but are not a runtime security boundary.

**2. Decimal input policy.** The validator accepts textual or existing `Decimal` values, excluding surprising conversions from floats and Booleans.

**3. Exact decimal behavior.** `Decimal("0.10") + Decimal("0.20")` is exactly `0.30` under the active context, unlike a typical binary floating-point representation.

**4. Explicit financial restrictions.** Negative, nonfinite, and unsupported representations are rejected before calculation.

**5. Quantization boundary.** Settlement-style rounding is a separate policy decision, not part of the language type-hint mechanism.

**6. Engineering implication.** A well-typed program can be financially incorrect; domain invariants and independently calculated reference values are mandatory.

## 2.6.4 Any, Object, Union, Literal, and Type Narrowing

### Theoretical Foundations

### Meaning of Common Annotations

- `Any`: a deliberate **escape hatch** from many static checks; it should be contained at I/O boundaries.
- `object`: any object is accepted, but the consumer must **narrow** before using type-specific operations.
- `int | str`: a union of permitted alternatives (PEP 604 syntax, Python 3.10+).
- `Literal["BUY", "SELL"]`: a static contract that distinguishes precise string values; runtime strings are not automatically restricted.
- `T | None`: an optional value that requires explicit treatment of `None`; zero is not missing.

`isinstance`, `type`, and `is None` tests can establish runtime information from which static checkers often narrow types. A `cast` is only a static assertion and does not check or convert the runtime value.

**Precise invariant:** `Any` can propagate unsafely, and `object` requires explicit runtime narrowing. Avoid `Any` in critical financial calculation internals.

In [4]:
from typing import Any, Literal, cast

TradeSide = Literal["BUY", "SELL"]

def normalize_side(raw: object) -> TradeSide:
    if raw == "BUY" or raw == "SELL":
        if isinstance(raw, str):
            return cast(TradeSide, raw)
    raise ValueError("Expected BUY or SELL")

assert normalize_side("BUY") == "BUY"
for invalid in ("SHORT", True, None, 1):
    try:
        normalize_side(invalid)
    except ValueError:
        print("Rejected side:", repr(invalid))

raw_quantity: object = 25
if type(raw_quantity) is int and raw_quantity > 0:
    quantity_typed: int = raw_quantity
else:
    raise ValueError("Invalid quantity")

risk_metric: Decimal | None = Decimal("0")
assert risk_metric is not None   # valid zero, not missing
unsafe_any: Any = "not a position"
claimed: int = cast(int, unsafe_any)
assert isinstance(claimed, str)  # cast has no runtime effect
print("Valid quantity:", quantity_typed)
print("cast retained original runtime type:", type(claimed).__name__)

Rejected side: 'SHORT'
Rejected side: True
Rejected side: None
Rejected side: 1
Valid quantity: 25
cast retained original runtime type: str


### **Analytical Interpretation**

**1. Literal contracts.** `Literal` lets a checker distinguish allowed string alternatives, but `normalize_side` must still reject invalid external strings at runtime.

**2. Boundary use of `cast`.** The cast is justified only after explicit runtime verification; a naked cast does not establish correctness.

**3. Narrowing from `object`.** The exact type test ensures a positive integer quantity and rejects the integer-like Boolean subtype.

**4. Optional metrics.** `Decimal("0")` must not be discarded using truthiness in place of an explicit missing-value check.

**5. `Any` unsoundness.** A cast applied to an `Any` value does not coerce the value. The runtime object remains a string.

**6. Financial consequence.** Use `object` at untrusted data boundaries and validate/narrow once; keep internal risk calculations fully specified.

## 2.6.5 Parameterized Collections, Read-Only Interfaces, and Variance

### Theoretical Foundations

### Built-in Generics

Python 3.9+ supports `list[T]`, `dict[K,V]`, `set[T]`, and `tuple[T,...]`. Annotations describe **element types**, but normal containers do not validate individual elements on mutation.

### Interface Choice

- `Sequence[T]`: read-only *interface* to an ordered collection; it does not guarantee the underlying object is immutable.
- `Mapping[K,V]`: read-only mapping *interface*, not necessarily an immutable object.
- `MutableSequence[T]` and `MutableMapping[K,V]`: express an intended mutable API.
- `tuple[T1,T2]`: a fixed-length heterogeneous tuple.
- `tuple[T,...]`: a variable-length homogeneous tuple.

### Variance and Mutation

Many mutable generic containers are invariant under static typing; covariant read-only APIs can be safer for consumers. Exact variance rules differ by abstraction, and should be checked against the typing specification.

### Runtime Generics

`list[int]` is a type annotation object but cannot generally be used as a parameterized runtime `isinstance()` test. Inspect the outer class and validate elements explicitly.

In [5]:
from collections.abc import Mapping, Sequence
from typing import get_args, get_origin

PriceWindow = tuple[Decimal, ...]
PriceLookup = Mapping[str, PriceWindow]

def compute_terminal_ratios(history: PriceLookup) -> dict[str, Decimal]:
    """Return end/start ratios for valid immutable price histories."""
    results: dict[str, Decimal] = {}
    for name, series in history.items():
        if len(series) < 2 or any(x <= 0 or not x.is_finite() for x in series):
            raise ValueError(f"Invalid price history for {name}")
        results[name] = series[-1] / series[0]
    return results

histories: dict[str, tuple[Decimal, ...]] = {
    "AAPL": (Decimal("100"), Decimal("105")),
    "MSFT": (Decimal("200"), Decimal("196")),
}
ratios = compute_terminal_ratios(histories)
assert ratios == {"AAPL": Decimal("1.05"), "MSFT": Decimal("0.98")}
assert get_origin(dict[str, Decimal]) is dict
assert get_args(dict[str, Decimal]) == (str, Decimal)

try:
    isinstance([1, 2], list[int])
except TypeError:
    print("Parameterized list cannot serve as isinstance target")

# A Mapping annotation alone does not freeze the supplied object.
mutable_mapping: dict[str, int] = {"AAPL": 100}
read_only_contract: Mapping[str, int] = mutable_mapping
mutable_mapping["AAPL"] = 110
assert read_only_contract["AAPL"] == 110
print("Read-only interface still sees underlying mutation")

Parameterized list cannot serve as isinstance target
Read-only interface still sees underlying mutation


### **Analytical Interpretation**

**1. Generic type declarations.** Parameterized collection annotations improve static tooling without changing the runtime behavior of built-in containers.

**2. Read-only contract vs immutable data.** A `Mapping` annotation limits intended API use, but a shared dictionary can still change through another alias.

**3. `get_origin` and `get_args`.** These tools inspect the structure of generic annotations. They do not validate the supplied market data.

**4. Parameterized runtime checks.** Runtime `isinstance()` checks of `list[int]` are not permitted; validate members explicitly.

**5. Domain validation.** The calculation verifies finite positive prices before division, a requirement unrelated to generics syntax.

**6. Interface design.** Accept the least-capable useful interface and return independent computed results to reduce mutation risk in research pipelines.

## 2.6.6 Type Aliases, NewType, and Financial Dimensional Analysis

### Theoretical Foundations

### Aliases and Nominal Distinctions

`TypeAlias` (Python 3.10+) identifies an alias in static analysis:

```python
QuoteMap: TypeAlias = Mapping[str, Decimal]
```

At runtime, this is just the aliased object. A Python 3.12+ `type QuoteMap = ...` statement creates a `TypeAliasType` and has different evaluation semantics.

`NewType("SecurityId", str)` signals a distinct **nominal static type** to checkers. At runtime, the returned callable performs no validation and typically returns its input unchanged. It is **not** a subclass and not an `isinstance` predicate.

### Dimensional Safety

Finance routinely mixes dollars, shares, rates, basis points, and native/converted currencies. Nominal types may help a checker detect accidental mixing **only if types are propagated consistently**. For robust runtime safety, use validated value objects or explicit unit fields.

In [6]:
from typing import NewType, TypeAlias

SecurityId = NewType("SecurityId", str)
USDNotional = NewType("USDNotional", Decimal)
BasisPoints = NewType("BasisPoints", Decimal)
PriceMap: TypeAlias = Mapping[SecurityId, Decimal]

security = SecurityId("AAPL")
notional = USDNotional(Decimal("15000.00"))
cost = BasisPoints(Decimal("8.0"))
assert type(security) is str
assert type(notional) is Decimal
assert type(cost) is Decimal

# Construct the well-defined economic conversion explicitly.
fees_usd = Decimal(notional) * Decimal(cost) / Decimal("10000")
assert fees_usd == Decimal("12.0000")
print("USD fees:", fees_usd)
print("NewType wraps without runtime conversion:", security, type(security).__name__)

# Intentional demonstration: NewType does not validate the payload at runtime.
unsafe_id = SecurityId(123)  # static checker should flag
assert unsafe_id == 123

USD fees: 12.000
NewType wraps without runtime conversion: AAPL str


### **Analytical Interpretation**

**1. Aliases document structure.** `PriceMap` makes APIs readable but does not create a new runtime collection class.

**2. Nominal typing is largely static.** `SecurityId`, `USDNotional`, and `BasisPoints` can help type checkers keep units distinct when used consistently.

**3. Runtime identity.** `NewType` constructors do not enforce their declared supertype; the intentionally bad `SecurityId(123)` demonstrates the limitation.

**4. Basis-point conversion.** Dividing by 10,000 is an explicit conversion between a basis-point count and a fractional rate.

**5. Higher-assurance design.** Validated dataclasses with explicit currencies/units provide stronger runtime guarantees than aliases or `NewType` alone.

**6. Financial consequence.** Distinct static units reduce certain programming errors, but they cannot validate market quotes or replace reconciliation.

## 2.6.7 TypedDict Contracts, Required Keys, and JSON Boundaries

### Theoretical Foundations

### Structured Dictionary Contracts

`TypedDict` describes the expected keys and types of a dictionary for static checkers. At runtime, a `TypedDict` instance is a **regular `dict`** and does not validate inputs.

- In Python 3.10, `total=False` and inheritance provide the portable mechanism for optional keys.
- Python 3.11 introduces `Required` and `NotRequired` for per-key control.
- Python 3.13 introduces `ReadOnly` for static read-only keys, without runtime freezing.

`__required_keys__` and `__optional_keys__` describe schema metadata, but their reliability can be affected by postponed annotations in older versions; avoid assuming them to be an unconditionally authoritative runtime validator.

### Input Boundary

Treat JSON-like payloads as `Mapping[str, object]`, not as already validated `TypedDict`. Validate required/extra keys and field values, then construct an explicitly typed internal record. Type declarations do not turn raw input into trusted data.

In [7]:
from typing import TypedDict

class CoreOrder(TypedDict):
    ticker: str
    quantity: int
    limit_price: str

class OrderWithOptionalNote(CoreOrder, total=False):
    note: str

valid_order: OrderWithOptionalNote = {
    "ticker": "AAPL",
    "quantity": 10,
    "limit_price": "185.25",
}
assert type(valid_order) is dict
assert OrderWithOptionalNote.__required_keys__ == frozenset({"ticker", "quantity", "limit_price"})
assert "note" in OrderWithOptionalNote.__optional_keys__

def validate_order_payload(raw: Mapping[str, object]) -> CoreOrder:
    expected = {"ticker", "quantity", "limit_price"}
    if not expected <= raw.keys() or not raw.keys() <= expected | {"note"}:
        raise ValueError("Unexpected or missing order fields")
    ticker = raw["ticker"]
    qty = raw["quantity"]
    price = raw["limit_price"]
    if not isinstance(ticker, str) or not ticker.strip():
        raise ValueError("Invalid ticker")
    if type(qty) is not int or qty <= 0:
        raise ValueError("Invalid quantity")
    if not isinstance(price, str):
        raise ValueError("Price must be decimal text")
    validated_decimal(price, strictly_positive=True)
    return {"ticker": ticker, "quantity": qty, "limit_price": price}

checked_order = validate_order_payload(valid_order)
assert checked_order["quantity"] == 10
for payload in ({"ticker": "AAPL", "quantity": True, "limit_price": "10"},
                {"ticker": "AAPL", "quantity": 1, "limit_price": "NaN"}):
    try:
        validate_order_payload(payload)
    except ValueError:
        print("Rejected invalid typed-dict-like payload")

Rejected invalid typed-dict-like payload
Rejected invalid typed-dict-like payload


### **Analytical Interpretation**

**1. Structural typing is static.** The declared dictionary structure is visible to a checker, but the runtime value is an ordinary mutable dictionary.

**2. Portable optional fields.** Inheritance with `total=False` works on Python 3.10 without requiring `NotRequired` from newer versions.

**3. Schema metadata.** Required and optional keys may be introspected for informational purposes; they must not substitute for an input validator.

**4. Explicit boundary validation.** Required fields, allowed extra fields, ticker spelling, quantity, and decimal price are all checked separately.

**5. Quantitative relevance.** Using a `TypedDict` directly as a promise about external JSON is unsound; a decoding layer is required for auditable trade data.

## 2.6.8 Annotated Metadata and Introspection

### Theoretical Foundations

### `Annotated[T, metadata...]`

`typing.Annotated` attaches metadata to a base type. Tools that do not recognize the metadata should treat the annotation as the underlying type `T`. **The metadata does not automatically enforce constraints.**

Use `get_type_hints(..., include_extras=True)` when analyzing trusted annotations and preserving metadata. `get_origin` and `get_args` can inspect the structure. `get_type_hints` may evaluate forward references or execute code; do not apply it indiscriminately to untrusted source.

### Documentation vs. Validation

A metadata marker such as `"USD"` or `"positive"` can aid schema generation, documentation, and a *separate* validation engine. It does not by itself check a `Decimal` instance.

In [8]:
from dataclasses import dataclass
from typing import Annotated, get_args, get_origin, get_type_hints

@dataclass(frozen=True)
class FinancialMetadata:
    unit: str
    strictly_positive: bool

MoneyAmount = Annotated[Decimal, FinancialMetadata("USD", True)]

def cost_model(capital: MoneyAmount, bps: Annotated[Decimal, "basis_points"]) -> Decimal:
    return capital * bps / Decimal("10000")

raw_metadata = get_type_hints(cost_model, include_extras=True)
capital_spec = raw_metadata["capital"]
assert get_origin(capital_spec) is Annotated
underlying, metadata = get_args(capital_spec)
assert underlying is Decimal
assert metadata == FinancialMetadata("USD", True)
assert get_type_hints(cost_model)["capital"] is Decimal
assert cost_model(Decimal("10000"), Decimal("10")) == Decimal("10")

# Annotations do not enforce the recorded positive constraint.
assert cost_model(Decimal("-10000"), Decimal("10")) == Decimal("-10")
print("Annotation metadata:", metadata)
print("Positive constraint requires explicit validation")

Annotation metadata: FinancialMetadata(unit='USD', strictly_positive=True)
Positive constraint requires explicit validation


### **Analytical Interpretation**

**1. Metadata preservation.** `include_extras=True` retains the complete `Annotated` form, including the domain metadata object.

**2. Default normalization.** `get_type_hints()` normally strips the metadata and reports the underlying `Decimal` type.

**3. Validation is separate.** The function still accepts negative capital and returns a negative cost; the metadata alone cannot prevent this.

**4. Safe introspection boundary.** Runtime type-hint resolution can evaluate annotations. Inspect only trusted code or use appropriate safer mechanisms.

**5. Quantitative use.** Metadata can drive explicit unit documentation, schema generation, or an independently written validator, but never claim implicit compliance.

## 2.6.9 Generic Algorithms, TypeVar, and ParamSpec Decorators

### Theoretical Foundations

### Generic Relationships

`TypeVar` preserves relationships between input and output types that `Any` would discard. `Generic` supports generic classes; `Callable` specifies callable signatures.

`ParamSpec` (Python 3.10+) captures a callable's parameter specification so a decorator can preserve the annotated signature. It does not ensure runtime argument validity.

### Variance and Protocols

A type parameter's variance may depend on the container/interface. A generic queue that accepts and returns values is generally invariant, while appropriate read-only interfaces may be covariant.

### Typed Decorators

`Callable[P, R] -> Callable[P, R]` plus `ParamSpec` describes a signature-preserving decorator; `functools.wraps` maintains key runtime metadata. The wrapper should avoid changing the input/output contract unexpectedly.

In [9]:
from collections.abc import Callable, Iterable
from functools import wraps
from typing import Generic, ParamSpec, TypeVar

T = TypeVar("T")
P = ParamSpec("P")
R = TypeVar("R")

def first_item(items: Iterable[T]) -> T:
    iterator = iter(items)
    return next(iterator)

class TypedBox(Generic[T]):
    def __init__(self, value: T) -> None:
        self.value = value
    def get(self) -> T:
        return self.value

assert first_item(["AAPL", "MSFT"]) == "AAPL"
assert TypedBox(Decimal("12.50")).get() == Decimal("12.50")

def traced_call(func: Callable[P, R]) -> Callable[P, R]:
    @wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        result = func(*args, **kwargs)
        return result
    return wrapper

@traced_call
def multiply_price(quantity: int, price: Decimal, *, fee: Decimal = Decimal("0")) -> Decimal:
    return quantity * price + fee

assert multiply_price(10, Decimal("11.20"), fee=Decimal("1.00")) == Decimal("113.00")
print("Wrapped function:", multiply_price.__name__)

Wrapped function: multiply_price


### **Analytical Interpretation**

**1. Type relationships.** `first_item` returns an element of the same static type as the iterable's elements, rather than erasing the relationship to `Any`.

**2. Generic storage.** `TypedBox[T]` expresses an intended relationship between construction and retrieval but does not validate unexpected runtime mutations automatically.

**3. `ParamSpec` forwarding.** The wrapper preserves argument and return annotations across positional, keyword-only, and optional parameters.

**4. Runtime semantics unchanged.** The financial multiplication still performs ordinary Python operations; incompatible input values can still raise exceptions.

**5. Engineering benefit.** Generic code and signature-preserving decorators support reusable research infrastructure without needlessly sacrificing static checking.

## 2.6.10 Structural Subtyping and Runtime-Checkable Protocols

### Theoretical Foundations

### `Protocol`

A Protocol specifies required **behavior** rather than inheritance from a particular concrete class. Static checkers can recognize structural compliance, including method argument and result types.

`@runtime_checkable` adds limited `isinstance()` support based primarily on the presence of the required members, **not** complete signature or value-type validation. A runtime protocol check is not a proof that a price provider returns a valid quote.

### Why It Matters

A typed valuation engine should depend on a narrow interface (e.g. `quote(ticker) -> Decimal`) rather than requiring an entire market-data class or broad mutable dictionary. Explicit contract validation is still needed at the boundary.

In [10]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class QuoteProvider(Protocol):
    def quote(self, ticker: str) -> Decimal: ...

class DictQuoteProvider:
    def __init__(self, values: Mapping[str, Decimal]) -> None:
        self.values = dict(values)
    def quote(self, ticker: str) -> Decimal:
        return self.values[ticker]

class DeceptiveQuoteProvider:
    def quote(self, ticker: str) -> str:
        return "not a decimal"  # violates static protocol contract

provider = DictQuoteProvider({"AAPL": Decimal("185.25")})
assert isinstance(provider, QuoteProvider)
assert provider.quote("AAPL") == Decimal("185.25")

# This limited runtime check can still return True despite a wrong return type.
deceptive_provider = DeceptiveQuoteProvider()
assert isinstance(deceptive_provider, QuoteProvider)
assert not isinstance(deceptive_provider.quote("AAPL"), Decimal)
print("Protocol membership is not return-type validation")

Protocol membership is not return-type validation


### **Analytical Interpretation**

**1. Structural contracts.** A provider can implement the expected interface without explicit inheritance from the Protocol class.

**2. Limited runtime checks.** Runtime-checkable protocols generally verify member presence, not that return values satisfy annotations.

**3. Deliberately inconsistent provider.** The deceptive provider passes the illustrated runtime membership check but returns the wrong type; static checkers should report incompatibility.

**4. Narrow dependency inversion.** Domain code should accept a protocol exposing only the operations it needs.

**5. Financial consequence.** Even with a statically typed provider, retrieved prices must be validated for finiteness, currency, and positivity before valuation.

## 2.6.11 Overload, TypeGuard, Cast, and Narrowing Contracts

### Theoretical Foundations

### Overloaded Functions

`@overload` provides multiple *static signatures* for one runtime implementation. Decorated overload declarations are not independently executed as alternative function bodies.

`TypeGuard[T]` (Python 3.10+) supplies a custom positive-branch narrowing rule to static checkers. It still behaves as an ordinary Boolean function at runtime; its accuracy depends on the implementation.

`TypeIs[T]` (Python 3.13+) supports narrowing in both branches under stronger compatibility conditions. It is not part of Python 3.10's standard library.

`cast(T, value)` does not transform or validate the object. Misuse can silence errors in a type checker while producing runtime failures.

In [11]:
from typing import TypeGuard, overload

@overload
def parse_quantity(value: str) -> int: ...
@overload
def parse_quantity(value: int) -> int: ...

def parse_quantity(value: str | int) -> int:
    if type(value) is int:
        result = value
    elif isinstance(value, str) and value.isdecimal():
        result = int(value)
    else:
        raise ValueError("Quantity must be a positive integer or digit string")
    if result <= 0:
        raise ValueError("Quantity must be positive")
    return result

assert parse_quantity("125") == 125
assert parse_quantity(25) == 25


def is_positive_int(value: object) -> TypeGuard[int]:
    return type(value) is int and value > 0

external_value: object = 42
if is_positive_int(external_value):
    checked_quantity: int = external_value
    assert checked_quantity == 42

for invalid in (True, "-10", 0):
    try:
        parse_quantity(invalid)  # type checker should reject bool/0-style misuse
    except ValueError:
        print("Rejected invalid quantity:", repr(invalid))

Rejected invalid quantity: True
Rejected invalid quantity: '-10'
Rejected invalid quantity: 0


### **Analytical Interpretation**

**1. Overloads are static descriptions.** Only the concrete implementation of `parse_quantity` executes.

**2. Runtime validation.** The function rejects invalid values regardless of which overload the checker selected.

**3. TypeGuard correctness.** The guard checks an exact `int` and strictly positive value, preventing Boolean-as-integer admission.

**4. Narrowing is conditional.** A checker can narrow an `object` after the positive branch of an accurately implemented guard.

**5. Unsupported later features.** `TypeIs` is deliberately discussed rather than imported unconditionally because the notebook targets Python 3.10+.

**6. Quantitative implication.** Overloads improve API ergonomics, but numerical data conversion policies should remain visible and independently tested.

## 2.6.12 Final, ClassVar, InitVar, and Dataclass Semantics

### Theoretical Foundations

### Static and Runtime Separation

- `Final[T]`: declares an intended static non-reassignment contract. It does **not** freeze a runtime binding.
- `ClassVar[T]`: marks a class-level variable for static analysis and tells `dataclasses` not to treat it as an instance field.
- `InitVar[T]`: creates an initialization-only dataclass parameter, passed to `__post_init__`, not a normal instance field.
- `@dataclass(frozen=True)`: blocks ordinary post-init attribute mutation. It is **not** automatically deeply immutable.

A frozen record containing a mutable list, dictionary, or user object can still exhibit mutable nested state. Construct independent immutable children when preserving a research snapshot.

In [12]:
from dataclasses import InitVar, dataclass, fields
from typing import ClassVar, Final

MAX_MODEL_VERSION: Final[int] = 3

@dataclass(frozen=True)
class ValidatedQuote:
    currency: str
    amount: Decimal
    source: InitVar[str]
    schema_version: ClassVar[int] = 3

    def __post_init__(self, source: str) -> None:
        if not isinstance(source, str) or not source.strip():
            raise ValueError("Source must be nonempty")
        if self.amount <= 0 or not self.amount.is_finite():
            raise ValueError("Quote must be strictly positive and finite")

quote_record = ValidatedQuote("USD", Decimal("185.25"), source="synthetic-feed")
assert quote_record.schema_version == 3
assert {f.name for f in fields(quote_record)} == {"currency", "amount"}
assert not hasattr(quote_record, "source")

from dataclasses import FrozenInstanceError
try:
    quote_record.amount = Decimal("200")
except FrozenInstanceError:
    print("Frozen quote rejected reassignment")

# Final is not runtime enforced at module scope; do not use mutation as a design pattern.
print("Static Final version declaration:", MAX_MODEL_VERSION)

Frozen quote rejected reassignment
Static Final version declaration: 3


### **Analytical Interpretation**

**1. `ClassVar` changes dataclass field handling.** The schema version is class metadata rather than an instance field.

**2. `InitVar` is initialization-only.** Source metadata is checked during construction but is not preserved as a standard instance field.

**3. Frozen dataclasses offer a runtime guard.** Ordinary reassignment raises `FrozenInstanceError`, unlike plain `Final` annotations.

**4. Deep mutability remains distinct.** A frozen object holding a list would not make that list immutable.

**5. Financial use.** Correctly designed records separate validated quote state, class-wide schema constants, and construction-time-only validation inputs.

## 2.6.13 AST Annotation Audits and Strict Static Checking

### Theoretical Foundations

### Code Inspection

`ast.AnnAssign` distinguishes annotated assignments from ordinary `ast.Assign`; `ast.arg.annotation` records function parameter annotations, and function definition nodes have return annotations. Source inspection does not prove that declared types match runtime values.

### Strict Static Checking

A static checker such as `mypy` or Pyright should be run **outside the standard-library execution path**; it is an optional developer dependency. Include a deliberate negative fixture for demonstration, but do **not** execute bad code or let the negative fixture enter production.

Illustrative project configuration:

```toml
[tool.mypy]
python_version = "3.10"
strict = true
warn_redundant_casts = true
warn_unused_ignores = true
```

Illustrative commands (terminal, with mypy separately installed):

```bash
python -m mypy --strict src/
python -m compileall -q src/
python -m unittest discover -v
```

### Boundary

Type checking is neither a proof of full correctness nor a substitute for financial domain validation. Annotations from untrusted source should be parsed or inspected without unsafe execution.

In [13]:
import ast

annotation_example = """
from decimal import Decimal
capital: Decimal = Decimal("100000")
missing: float

def calculate(value: Decimal, bps: int) -> Decimal:
    return value * bps / 10000
"""
syntax = ast.parse(annotation_example)
annotated_targets = [node for node in ast.walk(syntax) if isinstance(node, ast.AnnAssign)]
functions = [node for node in ast.walk(syntax) if isinstance(node, ast.FunctionDef)]
assert len(annotated_targets) == 2
assert {ast.unparse(node.target) for node in annotated_targets} == {"capital", "missing"}
assert len(functions) == 1
assert ast.unparse(functions[0].returns) == "Decimal"

print("Annotated assignments:")
for node in annotated_targets:
    print(" -", ast.unparse(node.target), "->", ast.unparse(node.annotation), "has value", node.value is not None)
print("Function returns:", ast.unparse(functions[0].returns))

# This is intentionally wrong code stored as text for a checker, never executed.
negative_type_fixture = 'quantity: int = "ten"\n'
assert isinstance(negative_type_fixture, str)

Annotated assignments:
 - capital -> Decimal has value True
 - missing -> float has value False
Function returns: Decimal


### **Analytical Interpretation**

**1. Static structure is observable.** The AST distinguishes annotations with values from annotation-only statements.

**2. Function contracts.** Parameter and return annotations can be inspected without running the function body.

**3. Static checker boundary.** The stored negative fixture is not executed; a checker should flag assigning a string literal to an integer annotation.

**4. `mypy` is optional.** The notebook runs on the Python standard library; using a separate checker in a project adds a useful second analysis layer.

**5. Research correctness.** Passing static analysis means the code satisfies only that checker's modeled type constraints, not that valuations or risk metrics are correct.

## 2.6.14 Practical Application — Typed Multi-Currency Portfolio Valuation and Risk Gate

### Research Problem

A quantitative research system receives holdings in **mixed native currencies**, price/FX data, a USD cash balance, and maximum-risk policies. A superficially well-typed calculation can still be wrong if it mixes native-currency amounts, accepts malformed holdings, or relies on mutable market data that later changes.

### Objective

Build a standard-library **type-safe, Decimal-based, immutable research pipeline** that:

1. Treats external payloads as untrusted `Mapping[str, object]` data.
2. Validates identifiers, currencies, sectors, integer quantities, and Decimal text.
3. Converts validated holdings into frozen typed domain records.
4. Accepts an FX provider through a narrow structural `Protocol`.
5. Validates FX availability and rates at runtime, even if the provider claims to be compliant.
6. Creates independent, immutable FX and holdings snapshots.
7. Computes native-currency and USD market values without premature rounding.
8. Calculates total positions, NAV, gross exposure, single-name weights, and sector weights.
9. Enforces three independent risk constraints with explicit violation records.
10. Provides deterministic ordering and referentially stable evaluation.
11. Identifies source mutations as separate dataset revisions rather than changing earlier results.
12. Verifies exact financial identities and rejection paths using automated tests.

### Types and Domain Model

```text
External Mapping[str, object] Holdings
                 |
                 v
        Explicit Runtime Decoder
                 |
                 v
         tuple[Holding, ...]
                 |
FXSnapshot --> Typed FX Provider Interface
                 |
                 v
        USD Valuation per Holding
                 |
                 v
      Portfolio NAV / Gross Exposure
                 |
                 v
    Risk Limits and Violation Records
                 |
                 v
      Immutable Valuation Report
```

### Financial Conventions

- Long-only positive holdings; no short borrow, derivatives, dividends, or financing costs.
- Native market value = shares × local price.
- USD market value = native market value × USD-per-one-unit-of-native-currency FX quote.
- NAV = cash USD + total USD market values.
- Gross exposure = sum of absolute USD position values; identical to aggregate USD position value for this long-only example.
- Single-name exposure ratio = absolute USD position value / NAV.
- Sector exposure ratio = sector aggregate USD position value / NAV.
- Gross exposure ratio = gross USD exposure / NAV.
- Risk rules breach only when a ratio is **strictly greater than** its limit.
- USD FX conversion rate must be exactly one.
- Market prices and FX rates must be finite and strictly positive.
- All amounts are computed using `Decimal`; display values are rounded only for presentation.
- The risk model does not estimate loss probability, correlation, VaR, or liquidity impact.

### Static vs. Runtime Contract

A typed `Holding` or `FxRateProvider` expresses *intent*. A trusted calculation must additionally validate real input values, including the type of the provider's returned quote. This project deliberately tests a deceptively shaped provider that violates its claimed return contract.

In [14]:
# ============================================================
# TYPED PORTFOLIO DOMAIN MODELS
# ============================================================

from collections.abc import Mapping, Sequence
from dataclasses import dataclass
from decimal import Decimal
from enum import Enum
from typing import Literal, NewType, Protocol

Ticker = NewType("Ticker", str)


class Currency(Enum):
    USD = "USD"
    EUR = "EUR"
    JPY = "JPY"
    GBP = "GBP"


class Sector(Enum):
    TECHNOLOGY = "TECHNOLOGY"
    AUTOMOTIVE = "AUTOMOTIVE"
    ENERGY = "ENERGY"


@dataclass(frozen=True, slots=True)
class Holding:
    """Validated long-only position in a native currency."""
    ticker: Ticker
    sector: Sector
    currency: Currency
    shares: int
    local_price: Decimal

    def __post_init__(self) -> None:
        if not isinstance(self.ticker, str) or not self.ticker.strip():
            raise ValueError("Ticker must be a nonempty string")
        if type(self.sector) is not Sector or type(self.currency) is not Currency:
            raise TypeError("Sector and currency must be enum members")
        if type(self.shares) is not int or self.shares <= 0:
            raise ValueError("Shares must be a strictly positive integer")
        if type(self.local_price) is not Decimal or not self.local_price.is_finite() or self.local_price <= 0:
            raise ValueError("Local price must be a positive finite Decimal")


@dataclass(frozen=True, slots=True)
class RiskLimits:
    """Maximum permitted ratios, expressed as unit fractions."""
    max_gross: Decimal
    max_single_name: Decimal
    max_sector: Decimal

    def __post_init__(self) -> None:
        for value in (self.max_gross, self.max_single_name, self.max_sector):
            if type(value) is not Decimal or not value.is_finite() or value <= 0 or value > 1:
                raise ValueError("Risk limits must be finite Decimal fractions in (0, 1]")


class UsdFxProvider(Protocol):
    def usd_per_unit(self, currency: Currency) -> Decimal: ...


@dataclass(frozen=True, slots=True)
class FxSnapshot:
    """Independent tuple-backed USD-per-native-unit exchange-rate snapshot."""
    rates: tuple[tuple[Currency, Decimal], ...]

    def __post_init__(self) -> None:
        seen: set[Currency] = set()
        for currency, rate in self.rates:
            if type(currency) is not Currency or currency in seen:
                raise ValueError("Invalid or duplicate FX currency")
            if type(rate) is not Decimal or not rate.is_finite() or rate <= 0:
                raise ValueError("FX rates must be positive finite Decimals")
            seen.add(currency)
        if Currency.USD not in seen or self.usd_per_unit(Currency.USD) != Decimal("1"):
            raise ValueError("USD FX rate must be exactly one")

    def usd_per_unit(self, currency: Currency) -> Decimal:
        for stored, rate in self.rates:
            if stored is currency:
                return rate
        raise KeyError(f"Missing FX rate: {currency.value}")


def make_fx_snapshot(raw_rates: Mapping[Currency, Decimal]) -> FxSnapshot:
    """Construct an independent FX snapshot from a mapping."""
    if not isinstance(raw_rates, Mapping):
        raise TypeError("FX input must be a mapping")
    return FxSnapshot(tuple(sorted(raw_rates.items(), key=lambda pair: pair[0].value)))


@dataclass(frozen=True, slots=True)
class ValuedHolding:
    holding: Holding
    native_market_value: Decimal
    usd_rate: Decimal
    usd_market_value: Decimal


RiskRule = Literal["GROSS", "SINGLE_NAME", "SECTOR"]


@dataclass(frozen=True, slots=True)
class RiskViolation:
    rule: RiskRule
    identifier: str
    observed_ratio: Decimal
    limit_ratio: Decimal


@dataclass(frozen=True, slots=True)
class ValuationReport:
    holdings: tuple[ValuedHolding, ...]
    cash_usd: Decimal
    positions_usd: Decimal
    nav_usd: Decimal
    gross_usd: Decimal
    sector_exposures: tuple[tuple[Sector, Decimal], ...]
    violations: tuple[RiskViolation, ...]

    @property
    def approved(self) -> bool:
        return not self.violations


print("Typed domain records and FX provider contract initialized")

Typed domain records and FX provider contract initialized


### **Analytical Interpretation**

**1. Domain concepts are explicit.** Enums distinguish native currencies and economic sectors rather than conflating them with arbitrary ticker strings.

**2. `NewType` is not enough.** `Ticker` improves checker-level clarity, but the `Holding` constructor still checks that the runtime ticker is a nonempty string.

**3. Protocol-driven integration.** The evaluator can depend on `UsdFxProvider` without importing or knowing a particular concrete FX implementation.

**4. Immutable boundaries.** `FxSnapshot` stores validated `(Currency, Decimal)` pairs in a tuple, so a later mutation of the source mapping cannot change existing FX quotes.

**5. Unit consistency.** Native and USD amounts occupy different named fields. No native price is treated as a USD price without an explicit conversion.

**6. Risk approval is derived.** `approved` is determined from the immutable violation collection, not a separately mutable Boolean that could become inconsistent.

**7. Model limits.** Static annotations and frozen records do not establish price-feed truth, FX timestamp synchronization, or portfolio-wide market-risk adequacy.

In [15]:
# ============================================================
# EXPLICIT INPUT DECODER / FINANCIAL VALUATION ENGINE
# ============================================================

_ALLOWED_HOLDING_KEYS = {"ticker", "sector", "currency", "shares", "local_price"}


def decode_holdings(raw_rows: Sequence[Mapping[str, object]]) -> tuple[Holding, ...]:
    """Convert untrusted rows into validated, independent frozen records."""
    if not isinstance(raw_rows, Sequence) or isinstance(raw_rows, (str, bytes)) or not raw_rows:
        raise ValueError("Nonempty holding row sequence required")

    decoded: list[Holding] = []
    seen: set[str] = set()
    for raw in raw_rows:
        if not isinstance(raw, Mapping) or set(raw.keys()) != _ALLOWED_HOLDING_KEYS:
            raise ValueError("Incorrect holding fields")

        ticker_value = raw["ticker"]
        sector_value = raw["sector"]
        currency_value = raw["currency"]
        shares_value = raw["shares"]
        local_price_value = raw["local_price"]

        if not isinstance(ticker_value, str) or not ticker_value.isascii() or not ticker_value.isalnum() or not 1 <= len(ticker_value) <= 12:
            raise ValueError("Ticker must be 1–12 ASCII alphanumeric characters")
        if ticker_value in seen:
            raise ValueError("Duplicate ticker")
        seen.add(ticker_value)
        if type(sector_value) is not str or type(currency_value) is not str:
            raise TypeError("Sector and currency must be strings")
        try:
            sector = Sector(sector_value)
            currency = Currency(currency_value)
        except ValueError as exc:
            raise ValueError("Unknown sector or currency") from exc
        if type(shares_value) is not int or shares_value <= 0:
            raise ValueError("Shares must be a positive integer (not bool)")
        local_price = validated_decimal(local_price_value, strictly_positive=True)
        decoded.append(Holding(Ticker(ticker_value), sector, currency, shares_value, local_price))

    return tuple(sorted(decoded, key=lambda h: h.ticker))


def value_portfolio(
    holdings: Sequence[Holding],
    cash_usd: Decimal,
    fx_provider: UsdFxProvider,
    limits: RiskLimits,
) -> ValuationReport:
    """Value a long-only multi-currency basket and enforce risk thresholds."""
    if not holdings:
        raise ValueError("At least one holding is required")
    if type(cash_usd) is not Decimal or not cash_usd.is_finite() or cash_usd < 0:
        raise ValueError("Cash must be a finite, nonnegative USD Decimal")
    if not isinstance(limits, RiskLimits):
        raise TypeError("Expected validated RiskLimits")

    seen: set[str] = set()
    valued: list[ValuedHolding] = []
    sectors: dict[Sector, Decimal] = {}

    for holding in sorted(holdings, key=lambda h: h.ticker):
        if not isinstance(holding, Holding):
            raise TypeError("Expected validated Holding")
        if holding.ticker in seen:
            raise ValueError("Duplicate security in valuation input")
        seen.add(holding.ticker)
        # Runtime validate provider outputs; a Protocol annotation is not sufficient.
        rate = fx_provider.usd_per_unit(holding.currency)
        if type(rate) is not Decimal or not rate.is_finite() or rate <= 0:
            raise ValueError("FX provider returned an invalid rate")
        if holding.currency is Currency.USD and rate != Decimal("1"):
            raise ValueError("USD conversion rate must be exactly one")
        native_value = Decimal(holding.shares) * holding.local_price
        usd_value = native_value * rate
        if not native_value.is_finite() or not usd_value.is_finite():
            raise OverflowError("Nonfinite valuation")
        valued.append(ValuedHolding(holding, native_value, rate, usd_value))
        sectors[holding.sector] = sectors.get(holding.sector, Decimal("0")) + usd_value

    positions_usd = sum((v.usd_market_value for v in valued), Decimal("0"))
    nav_usd = cash_usd + positions_usd
    gross_usd = sum((abs(v.usd_market_value) for v in valued), Decimal("0"))
    if not nav_usd.is_finite() or nav_usd <= 0:
        raise ValueError("NAV must be finite and strictly positive")

    breaches: list[RiskViolation] = []
    gross_ratio = gross_usd / nav_usd
    if gross_ratio > limits.max_gross:
        breaches.append(RiskViolation("GROSS", "PORTFOLIO", gross_ratio, limits.max_gross))

    for item in valued:
        weight = abs(item.usd_market_value) / nav_usd
        if weight > limits.max_single_name:
            breaches.append(RiskViolation("SINGLE_NAME", item.holding.ticker, weight, limits.max_single_name))

    for sector in sorted(sectors, key=lambda s: s.value):
        sector_ratio = sectors[sector] / nav_usd
        if sector_ratio > limits.max_sector:
            breaches.append(RiskViolation("SECTOR", sector.value, sector_ratio, limits.max_sector))

    return ValuationReport(
        holdings=tuple(valued),
        cash_usd=cash_usd,
        positions_usd=positions_usd,
        nav_usd=nav_usd,
        gross_usd=gross_usd,
        sector_exposures=tuple(sorted(sectors.items(), key=lambda pair: pair[0].value)),
        violations=tuple(breaches),
    )

print("Strict input decoder and USD risk evaluator initialized")

Strict input decoder and USD risk evaluator initialized


### **Analytical Interpretation**

**1. External values enter as `object`.** The decoder does not assume that a `TypedDict` or function signature makes external data trustworthy.

**2. Domain-specific validation.** Shares must be positive exact integers; price text must be finite and positive; tickers, currencies, sectors, and duplicates are checked explicitly.

**3. Typed internal state.** The decoder returns a tuple of frozen `Holding` objects. Changes to the original row dictionaries cannot change these records.

**4. FX provider separation.** A structural protocol describes the method expected from the provider; each returned value is still checked for correct runtime type and economic range.

**5. Explicit currency conversion.** Native notional is multiplied by the quote interpreted as USD per one native unit, keeping conversion direction auditable.

**6. Independent reconciliation.** Position value, NAV, gross value, and sector subtotals are computed in explicit variables before returning a frozen report.

**7. Strict thresholds.** Ratios breach only above their limits, not at equality. A dedicated test must verify this boundary.

**8. Research scope.** The code is a long-only snapshot valuation, not a dynamic trading strategy, VaR model, or portfolio performance backtest.

In [16]:
# ============================================================
# MULTI-CURRENCY VALUATION EXAMPLE
# ============================================================

raw_holdings: list[dict[str, object]] = [
    {"ticker": "AAPL", "sector": "TECHNOLOGY", "currency": "USD", "shares": 100, "local_price": "190.00"},
    {"ticker": "SAP", "sector": "TECHNOLOGY", "currency": "EUR", "shares": 80, "local_price": "150.00"},
    {"ticker": "TM", "sector": "AUTOMOTIVE", "currency": "JPY", "shares": 50, "local_price": "3000.00"},
    {"ticker": "BP", "sector": "ENERGY", "currency": "GBP", "shares": 200, "local_price": "5.20"},
]
raw_fx: dict[Currency, Decimal] = {
    Currency.USD: Decimal("1"),
    Currency.EUR: Decimal("1.10"),
    Currency.JPY: Decimal("0.0067"),
    Currency.GBP: Decimal("1.27"),
}
portfolio_holdings = decode_holdings(raw_holdings)
original_fx = make_fx_snapshot(raw_fx)
original_limits = RiskLimits(
    max_gross=Decimal("0.24"),
    max_single_name=Decimal("0.12"),
    max_sector=Decimal("0.22"),
)
report = value_portfolio(portfolio_holdings, Decimal("100000"), original_fx, original_limits)

print("=" * 82)
print("TYPED MULTI-CURRENCY PORTFOLIO VALUATION")
print("=" * 82)
print(f"{'Ticker':<10}{'Currency':<12}{'Native Value':>18}{'USD Rate':>18}{'USD Value':>20}")
for item in report.holdings:
    print(f"{item.holding.ticker:<10}{item.holding.currency.value:<12}{item.native_market_value:>18,.2f}{item.usd_rate:>18}{item.usd_market_value:>20,.2f}")
print("-" * 82)
print(f"Cash (USD): ${report.cash_usd:,.2f}")
print(f"Position Value (USD): ${report.positions_usd:,.2f}")
print(f"NAV (USD): ${report.nav_usd:,.2f}")
print(f"Gross/NAV: {report.gross_usd / report.nav_usd:.4%}")
print("Risk approval:", report.approved)
for violation in report.violations:
    print(f"BREACH {violation.rule:<12} {violation.identifier:<12} {violation.observed_ratio:.4%} > {violation.limit_ratio:.2%}")

# Exact reference values (Decimal, no premature cent rounding).
assert report.positions_usd == Decimal("34525.8000")
assert report.nav_usd == Decimal("134525.8000")
assert report.gross_usd == report.positions_usd
assert len(report.violations) == 3
assert {(v.rule, v.identifier) for v in report.violations} == {
    ("GROSS", "PORTFOLIO"),
    ("SINGLE_NAME", "AAPL"),
    ("SECTOR", "TECHNOLOGY"),
}

# A source revision must not mutate the already validated portfolio snapshot.
raw_holdings[0]["local_price"] = "9999.00"
raw_fx[Currency.EUR] = Decimal("2.00")
assert portfolio_holdings[0].local_price == Decimal("190.00")
assert original_fx.usd_per_unit(Currency.EUR) == Decimal("1.10")
assert value_portfolio(portfolio_holdings, Decimal("100000"), original_fx, original_limits) == report
print("Source mutation isolation: PASSED")

TYPED MULTI-CURRENCY PORTFOLIO VALUATION
Ticker    Currency          Native Value          USD Rate           USD Value
AAPL      USD                  19,000.00                 1           19,000.00
BP        GBP                   1,040.00              1.27            1,320.80
SAP       EUR                  12,000.00              1.10           13,200.00
TM        JPY                 150,000.00            0.0067            1,005.00
----------------------------------------------------------------------------------
Cash (USD): $100,000.00
Position Value (USD): $34,525.80
NAV (USD): $134,525.80
Gross/NAV: 25.6648%
Risk approval: False
BREACH GROSS        PORTFOLIO    25.6648% > 24.00%
BREACH SINGLE_NAME  AAPL         14.1237% > 12.00%
BREACH SECTOR       TECHNOLOGY   23.9359% > 22.00%
Source mutation isolation: PASSED


### Expected Results — Financial Reference Table

| Security | Native Currency | Native Market Value | USD Rate per Native Unit | USD Market Value |
|---|---|---:|---:|---:|
| AAPL | USD | 19,000.00 | 1 | 19,000.00 |
| SAP | EUR | 12,000.00 | 1.10 | 13,200.00 |
| TM | JPY | 150,000.00 | 0.0067 | 1,005.00 |
| BP | GBP | 1,040.00 | 1.27 | 1,320.80 |

**Position Value:** $34,525.80  
**Cash:** $100,000.00  
**NAV:** $134,525.80

**Three expected breaches:** gross exposure > 24% NAV; AAPL single-name exposure > 12% NAV; Technology sector exposure > 22% NAV.

### **Analytical Interpretation**

**1. Explicit currency conversion.** Every native position is converted to USD using a clearly defined quote direction. Mixing GBP or JPY native values directly into USD NAV would be a financial error not caught by generic numeric annotations alone.

**2. Exact reconciliation.** Total position USD market value equals the sum of the four converted holdings; NAV equals cash plus those positions. The example deliberately delays display rounding.

**3. Sector and concentration logic.** The technology sector combines AAPL and SAP after converting both to USD; the exposure rules therefore compare economically meaningful common-currency values.

**4. Static types vs domain validation.** Annotated domain records help catch mistakes in development; runtime validation is what rejects malformed observations and deceptive FX providers.

**5. Snapshot isolation.** Changing the original raw holding and FX dictionaries does not change the already decoded holdings or frozen FX snapshot.

**6. Auditability and reproducibility.** The report is derived from explicit immutable inputs, sorted holdings, and deterministic calculations, making independent result verification possible.

**7. Limitations.** FX/price timestamp alignment, stale-quote detection, settlement rounding, corporate actions, bid/ask spread, and leverage financing would require a more complete production system.

## 2.6.15 Automated Testing and Financial Invariants

### Testing Objective

The test suite must verify language-level typing limitations and independent economic identities. Static type assertions do not replace numerical tests, and runtime tests cannot prove that a static checker accepts or rejects every program.

### Core Invariants

**I1 — Input isolation.** Mutating external rows or FX mappings after decoding does not modify existing typed snapshots.

**I2 — Native-value identity.** `NativeMarketValue = Shares × LocalPrice`.

**I3 — Currency-conversion identity.** `USDMarketValue = NativeMarketValue × USDPerNativeUnit`.

**I4 — NAV conservation.** `NAV = CashUSD + Sum(USDMarketValues)`.

**I5 — Gross-exposure identity.** `GrossUSD = Sum(abs(USDMarketValue_i))`.

**I6 — Sector reconciliation.** `Sum(SectorExposuresUSD) = Sum(USDMarketValues)` for the long-only example.

**I7 — Correct rejection.** Missing FX, nonfinite FX, NaN prices, unexpected fields, Boolean share quantities, unsupported currencies/sectors, and duplicate tickers are rejected.

**I8 — Threshold boundary.** A ratio exactly equal to a configured limit does not breach; a greater ratio does.

**I9 — Determinism.** Changing input mapping order without changing content does not change the financial report.

**I10 — Structural typing limitation.** A provider may satisfy a runtime protocol membership test yet return invalid content; the financial evaluator must reject that value.

**I11 — No premature conversion.** Decimal text is preserved without conversion from binary float.

**I12 — No silent name-level type enforcement.** Python annotations cannot replace runtime input validation.

In [17]:
# ============================================================
# AUTOMATED CONTRACT, FINANCIAL, AND BOUNDARY TESTS
# ============================================================

import unittest


def sample_rows() -> list[dict[str, object]]:
    return [
        {"ticker": "AAPL", "sector": "TECHNOLOGY", "currency": "USD", "shares": 100, "local_price": "190.00"},
        {"ticker": "SAP", "sector": "TECHNOLOGY", "currency": "EUR", "shares": 80, "local_price": "150.00"},
        {"ticker": "TM", "sector": "AUTOMOTIVE", "currency": "JPY", "shares": 50, "local_price": "3000.00"},
        {"ticker": "BP", "sector": "ENERGY", "currency": "GBP", "shares": 200, "local_price": "5.20"},
    ]


def sample_fx() -> dict[Currency, Decimal]:
    return {
        Currency.USD: Decimal("1"), Currency.EUR: Decimal("1.10"),
        Currency.JPY: Decimal("0.0067"), Currency.GBP: Decimal("1.27"),
    }


class TypeSafeQuantResearchTests(unittest.TestCase):
    def setUp(self) -> None:
        self.raw = sample_rows()
        self.fx_raw = sample_fx()
        self.holdings = decode_holdings(self.raw)
        self.fx = make_fx_snapshot(self.fx_raw)
        self.limits = RiskLimits(Decimal("0.24"), Decimal("0.12"), Decimal("0.22"))
        self.result = value_portfolio(self.holdings, Decimal("100000"), self.fx, self.limits)

    def test_reference_positions(self):
        self.assertEqual(self.result.positions_usd, Decimal("34525.8000"))

    def test_reference_nav(self):
        self.assertEqual(self.result.nav_usd, Decimal("134525.8000"))

    def test_reference_risk_rejection(self):
        self.assertFalse(self.result.approved)
        self.assertEqual(len(self.result.violations), 3)

    def test_exact_valuation_by_ticker(self):
        observed = {x.holding.ticker: x.usd_market_value for x in self.result.holdings}
        self.assertEqual(observed["AAPL"], Decimal("19000.00"))
        self.assertEqual(observed["SAP"], Decimal("13200.0000"))
        self.assertEqual(observed["TM"], Decimal("1005.000000"))
        self.assertEqual(observed["BP"], Decimal("1320.8000"))

    def test_nav_identity(self):
        self.assertEqual(self.result.nav_usd, self.result.cash_usd + sum((x.usd_market_value for x in self.result.holdings), Decimal("0")))

    def test_gross_identity(self):
        self.assertEqual(self.result.gross_usd, sum((abs(x.usd_market_value) for x in self.result.holdings), Decimal("0")))

    def test_sector_reconciliation(self):
        self.assertEqual(sum((v for _, v in self.result.sector_exposures), Decimal("0")), self.result.positions_usd)

    def test_native_price_identity(self):
        for v in self.result.holdings:
            self.assertEqual(v.native_market_value, v.holding.local_price * v.holding.shares)

    def test_fx_conversion_identity(self):
        for v in self.result.holdings:
            self.assertEqual(v.usd_market_value, v.native_market_value * v.usd_rate)

    def test_cash_nonnegative(self):
        with self.assertRaises(ValueError):
            value_portfolio(self.holdings, Decimal("-1"), self.fx, self.limits)

    def test_missing_fx(self):
        partial = make_fx_snapshot({Currency.USD: Decimal("1")})
        with self.assertRaises(KeyError):
            value_portfolio(self.holdings, Decimal("100000"), partial, self.limits)

    def test_invalid_fx_zero(self):
        rates = sample_fx()
        rates[Currency.EUR] = Decimal("0")
        with self.assertRaises(ValueError):
            make_fx_snapshot(rates)

    def test_invalid_fx_nan(self):
        rates = sample_fx()
        rates[Currency.EUR] = Decimal("NaN")
        with self.assertRaises(ValueError):
            make_fx_snapshot(rates)

    def test_invalid_usd_rate(self):
        rates = sample_fx()
        rates[Currency.USD] = Decimal("1.01")
        with self.assertRaises(ValueError):
            make_fx_snapshot(rates)

    def test_duplicate_fx_currency(self):
        with self.assertRaises(ValueError):
            FxSnapshot(((Currency.USD, Decimal("1")), (Currency.USD, Decimal("1"))))

    def test_deceptive_provider(self):
        with self.assertRaises(ValueError):
            value_portfolio(self.holdings, Decimal("100000"), DeceptiveFxProvider(), self.limits)

    def test_duplicate_ticker(self):
        rows = sample_rows()
        rows.append(rows[0].copy())
        with self.assertRaises(ValueError):
            decode_holdings(rows)

    def test_boolean_quantity(self):
        rows = sample_rows()
        rows[0]["shares"] = True
        with self.assertRaises(ValueError):
            decode_holdings(rows)

    def test_float_price_rejected(self):
        rows = sample_rows()
        rows[0]["local_price"] = 190.0
        with self.assertRaises(TypeError):
            decode_holdings(rows)

    def test_nonfinite_price(self):
        rows = sample_rows()
        rows[0]["local_price"] = "NaN"
        with self.assertRaises(ValueError):
            decode_holdings(rows)

    def test_unknown_sector(self):
        rows = sample_rows()
        rows[0]["sector"] = "UNKNOWN"
        with self.assertRaises(ValueError):
            decode_holdings(rows)

    def test_unknown_currency(self):
        rows = sample_rows()
        rows[0]["currency"] = "AUD"
        with self.assertRaises(ValueError):
            decode_holdings(rows)

    def test_unexpected_field(self):
        rows = sample_rows()
        rows[0]["untrusted"] = "payload"
        with self.assertRaises(ValueError):
            decode_holdings(rows)

    def test_snapshot_data_isolation(self):
        baseline = self.result
        self.raw[0]["local_price"] = "999.99"
        self.fx_raw[Currency.EUR] = Decimal("5.00")
        self.assertEqual(self.fx.usd_per_unit(Currency.EUR), Decimal("1.10"))
        self.assertEqual(value_portfolio(self.holdings, Decimal("100000"), self.fx, self.limits), baseline)

    def test_determinism_with_reversed_rows(self):
        alternate = decode_holdings(list(reversed(sample_rows())))
        alt_report = value_portfolio(alternate, Decimal("100000"), self.fx, self.limits)
        self.assertEqual(self.result, alt_report)

    def test_relaxed_limits_approve(self):
        relaxed = RiskLimits(Decimal("0.40"), Decimal("0.20"), Decimal("0.30"))
        approved = value_portfolio(self.holdings, Decimal("100000"), self.fx, relaxed)
        self.assertTrue(approved.approved)
        self.assertEqual(approved.violations, ())

    def test_exact_threshold_boundary(self):
        gross_ratio = self.result.gross_usd / self.result.nav_usd
        max_single = max(x.usd_market_value / self.result.nav_usd for x in self.result.holdings)
        max_sector = max(value / self.result.nav_usd for _, value in self.result.sector_exposures)
        boundary = RiskLimits(gross_ratio, max_single, max_sector)
        self.assertTrue(value_portfolio(self.holdings, Decimal("100000"), self.fx, boundary).approved)

    def test_invalid_risk_limit(self):
        with self.assertRaises(ValueError):
            RiskLimits(Decimal("-0.1"), Decimal("0.2"), Decimal("0.3"))

    def test_portfolio_inputs_are_frozen(self):
        from dataclasses import FrozenInstanceError
        with self.assertRaises(FrozenInstanceError):
            self.holdings[0].shares = 777

    def test_optional_zero_value(self):
        self.assertIsNotNone(Decimal("0"))
        self.assertEqual(validated_decimal("0"), Decimal("0"))

    def test_typed_dict_is_regular_dictionary(self):
        self.assertIs(type(valid_order), dict)

    def test_protocol_check_not_return_validation(self):
        self.assertTrue(isinstance(deceptive_provider, QuoteProvider))
        self.assertIsInstance(deceptive_provider.quote("AAPL"), str)

    def test_type_guard_rejects_bool(self):
        self.assertFalse(is_positive_int(True))
        self.assertTrue(is_positive_int(7))

    def test_decimal_float_vs_text(self):
        self.assertNotEqual(Decimal(0.1), Decimal("0.1"))

    def test_annotated_metadata_not_constraint(self):
        self.assertEqual(cost_model(Decimal("-10000"), Decimal("10")), Decimal("-10"))


class DeceptiveFxProvider:
    """Has the right method name but returns an invalid runtime value."""
    def usd_per_unit(self, currency: Currency) -> str:
        return "invalid"


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TypeSafeQuantResearchTests)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "A type contract or financial invariant test failed"
print("All contract and numerical verification tests passed")

test_annotated_metadata_not_constraint (__main__.TypeSafeQuantResearchTests.test_annotated_metadata_not_constraint) ... 

ok


test_boolean_quantity (__main__.TypeSafeQuantResearchTests.test_boolean_quantity) ... 

ok


test_cash_nonnegative (__main__.TypeSafeQuantResearchTests.test_cash_nonnegative) ... 

ok


... (62 outputs left out)

test_unknown_sector (__main__.TypeSafeQuantResearchTests.test_unknown_sector) ... 

ok


----------------------------------------------------------------------
Ran 35 tests in 0.020s

OK


All contract and numerical verification tests passed


### **Analytical Interpretation**

**1. Reference financial metrics.** The tests independently fix expected portfolio USD values and reconcile cash, NAV, gross exposure, sector totals, and native-to-USD conversions.

**2. Runtime rejection paths.** Invalid price types, NaN values, unknown sector/currency codes, missing FX rates, duplicate securities, and malformed schema keys are rejected.

**3. Unit boundaries.** Quote directions and reporting currencies are explicit: an FX provider returning a malformed value fails despite its method being structurally present.

**4. Financial limit equality.** The exact-threshold test verifies that the specified `>` policy treats equality as compliant, an important boundary in concentration limits.

**5. Snapshot isolation and reproducibility.** Modifying the original raw inputs cannot change decoded holdings, FX snapshot, or valuation report. Reversing source-row order leaves the report unchanged.

**6. Type-system boundaries.** A valid `TypedDict` is still a dictionary; a `TypeGuard` is runtime Boolean logic; `Annotated` metadata is not enforcement; a runtime-checkable protocol cannot validate returned financial values.

**7. Limits of testing.** These checks do not establish statistical significance, market-data authenticity, production transaction correctness, or operational availability.

**8. Engineering conclusion.** Correct typing reduces a class of mistakes; exact domain validation and numerical reconciliation establish separate, essential assurance layers.

## 2.6.16 Final Analytical Interpretation

### 1. The Meaning of an Annotation

An annotation communicates an intended static contract. It is not a runtime type declaration that permanently constrains a name.

### 2. Annotated Assignment vs. Initialization

An annotation without an assigned value need not create a runtime value. In function scope it can still classify a name as local, affecting resolution and potentially raising `UnboundLocalError`.

### 3. Cross-Version Semantics

The 3.10–3.13 eager and future-postponed models differ from the Python 3.14+ default lazy evaluation model. Runtime annotation inspection must be designed for the target interpreter version.

### 4. Static Types and Valid Data Are Separate

A program can satisfy a static type checker while accepting a negative price, incorrect currency, stale market quote, or economically nonsensical leverage assumption.

### 5. Safe Input Boundaries

Use `object` and explicit validation when processing untrusted data. Avoid allowing `Any` to flow unchecked into financial calculations.

### 6. Optional Values

The distinction between `None` and a legitimate zero is fundamental in financial metrics. Truthiness checks should not conflate missing data and zero exposure.

### 7. Generic Collection Design

`Mapping` and `Sequence` express API capabilities, not immutability. To preserve historical observations, construct independent immutable snapshot contents.

### 8. Nominal Types and Financial Units

`NewType` can represent conceptual distinctions between shares, rates, and monetary values for static tools. Its constructors do not perform runtime validation or conversion.

### 9. Structured Records

`TypedDict` provides a useful dictionary schema to checkers but is an ordinary dictionary at runtime. Decode and validate external payloads before use.

### 10. Metadata and Introspection

`Annotated` can preserve domain annotations for documentation and tool-specific handling. Validation requires a separate mechanism. Resolving untrusted type hints can be unsafe.

### 11. Generic Algorithms and Higher-Order Functions

`TypeVar`, `Generic`, and `ParamSpec` encode useful relationships and preserve function signatures across wrappers, improving reusable quantitative software architecture.

### 12. Structural Subtyping

`Protocol` supports dependency inversion through behavior-oriented interfaces. Runtime protocol checks are intentionally shallow and do not prove numeric correctness.

### 13. Overloads and Type Narrowing

`@overload`, `TypeGuard`, and `cast` inform static analyses. Their reliability depends on accurate implementations and justified assertions.

### 14. Dataclass Contracts

`ClassVar`, `InitVar`, `Final`, and `frozen=True` have distinct meanings. Frozen records restrict ordinary reassignment but require immutable nested data for strong snapshot isolation.

### 15. Multi-Currency Financial Reconciliation

The practical project explicitly converts each security from native currency to USD, reconciles cash and positions to NAV, and evaluates exposure ratios in a common unit.

### 16. Data Integrity and Research Reproducibility

External source mutation does not change previously captured typed holdings or FX snapshots. Deterministic sorted evaluation supports repeated numerical verification.

### 17. Model and Engineering Limits

The example excludes bid-ask spreads, settlement, financing, derivatives, corporate actions, timestamp alignment, and distributed transaction processing. It also does not replace optional strict static checking in CI.

### Final Assessment

Advanced Python type-system proficiency requires precise knowledge of **what an annotation tells a checker**, **what Python does at runtime**, and **which financial invariants must be independently verified**. A robust quantitative pipeline integrates all three rather than treating any one layer as sufficient.

## Advanced Practice Exercises — Expert Level

### Exercise 1: Annotation Scope and Compiler Classification

**Objective:** Build an AST/symbol-table auditor for module, class, and function annotations.

**Tasks:** Distinguish annotation-only from value-bearing assignments; inspect `AnnAssign`, parameter annotations, and return annotations; demonstrate `UnboundLocalError` from a function-local annotation; evaluate version-specific annotation introspection safely; generate source-location reports.

**Requirements:** Python 3.10+; no execution of untrusted code; independent tests; explain annotation-storage differences in 3.14+.

---

### Exercise 2: Static Type Contracts Versus Economic Runtime Invariants

**Objective:** Design an input decoder for real-world financial records with inaccurate type hints and malformed data.

**Tasks:** Separate external `Mapping[str, object]` from internal validated records; reject bool quantities, NaN prices, negative fees, unsupported currencies, duplicate keys, and unexpected fields; construct immutable domain objects; compare checker errors with runtime error paths.

**Requirements:** No implicit float-to-Decimal conversion; explicit financial units; boundary-condition tests.

---

### Exercise 3: Generic Research API Design

**Objective:** Create reusable typed algorithms without leaking `Any`.

**Tasks:** Implement `TypeVar`-preserving selection, `Generic` result containers, and `ParamSpec` wrappers; inspect immutable vs mutable collection APIs; provide negative static fixtures for invalid assignments and incorrect callback signatures; document variance assumptions.

**Requirements:** Preserve callable argument signatures; do not assert runtime safety from generic annotations.

---

### Exercise 4: Protocol-Based Market Data Architecture

**Objective:** Support several price sources behind one structural interface.

**Tasks:** Implement in-memory, cached, and deliberately malformed providers; validate outputs; test missing symbols, stale timestamps, negative prices, and unavailable FX quotes; contrast `@runtime_checkable` structural membership with a full financial validation contract.

**Requirements:** Narrow public interface; deterministic tests; document trust boundaries.

---

### Exercise 5: Advanced `TypedDict` and Metadata Schema System

**Objective:** Model evolving event payload schemas across Python versions.

**Tasks:** Use portable required/optional-field definitions; demonstrate Python 3.11 `NotRequired` and 3.13 `ReadOnly` using version-gated examples; preserve `Annotated` metadata; generate safe human-readable schemas; avoid evaluating annotations from untrusted source text.

**Requirements:** Version-aware behavior; explicit runtime schema checks; test extra/missing keys.

---

### Exercise 6: Nominal Unit Safety and Decimal Reconciliation

**Objective:** Prevent accidental mixing of price, quantity, native notional, USD notional, return fraction, and basis points.

**Tasks:** Introduce `NewType` identifiers, validated runtime value objects, currency enums, and `Decimal` quantization policies; implement FX conversion; compare conversion-before-aggregation versus aggregation-before-conversion; verify financial identities.

**Requirements:** Explicit FX direction; no direct `Decimal(float)`; independently derived reference cases.

---

### Exercise 7: Runtime Annotation Introspection Security Review

**Objective:** Investigate the threat model of annotation introspection.

**Tasks:** Compare `__annotations__`, `inspect.get_annotations`, and `typing.get_type_hints`; document code execution risks; inspect trusted source using AST; demonstrate deferred evaluation in a version-gated way; produce a safe API design for documentation generation.

**Requirements:** Never execute untrusted source or references; include Python 3.14+ considerations.

---

### Exercise 8: Overload and Type Narrowing Correctness

**Objective:** Build a checked API supporting multiple input representations.

**Tasks:** Specify overload signatures; implement validation and parsing; introduce a deliberately incorrect `TypeGuard` and discuss how it can mislead a checker; compare `cast` with real validation; add strict-checker negative fixtures.

**Requirements:** Validate positive integer quantities and explicit `None`; avoid using casts to silence legitimate errors.

---

### Exercise 9: Immutable Multi-Currency Valuation Snapshots

**Objective:** Extend the main project to dated financial observations.

**Tasks:** Add quote availability time, timezone-aware timestamps, stale-rate policies, position-level market-value changes, immutable snapshots, deterministic serialization, and revision sensitivity; implement reference reconciliation at each snapshot date.

**Requirements:** No look-ahead leakage; explicit asynchronous data timing assumptions; numerical and temporal invariants.

---

### Exercise 10: Automated Type Quality Gates

**Objective:** Integrate optional strict static checking and runtime testing into a reproducible GitHub project.

**Tasks:** Extract source from the notebook into package modules; build `pyproject.toml`; add strict `mypy`/Pyright configurations; create positive/negative typing fixtures; integrate runtime tests and static checks into CI; document interpreter-version support.

**Requirements:** Keep optional tools separate from the notebook's standard-library execution path; preserve all financial reference outputs.

---

### Exercise 11: Portfolio Risk Policy Boundary Testing

**Objective:** Strengthen exposure constraints under edge cases.

**Tasks:** Test exactly-at-limit and just-above-limit positions; modify cash/FX independently; test concentration across currencies and sectors; introduce zero-cash portfolios and extremely small FX rates; verify NAV, gross, sector, and single-name identities.

**Requirements:** `Decimal` computations; reproducible and independent reference values; explicit strict-inequality policies.

---

### Exercise 12: Typed Backtesting Feature Interface

**Objective:** Develop a statically typed feature pipeline that respects point-in-time availability.

**Tasks:** Specify Protocol interfaces for feature providers, immutable typed samples, `TypedDict` ingestion, missing-data optional types, and time-aware feature records; verify that outcome labels are never available to earlier features; explain why strong types cannot alone prove temporal correctness.

**Requirements:** Separation of static correctness, runtime validation, and statistical methodology; synthetic deterministic data.

## Advanced Challenge — Production-Oriented Typed Financial Research Library

### Objective

Build a reusable and independently auditable typed quantitative research library that combines Python's formal annotation semantics, generic software architecture, secure input validation, and multi-currency financial reconciliation.

### Required Components

1. **Typed Domain Layer:** `Currency`, `SecurityId`, prices, quantities, native notionals, USD notionals, and risk-limit records; explain use of nominal versus structural types.
2. **Input Decoder:** Strict JSON-like payload validation and schema versioning; reject invalid identifiers, malformed quantities, and nonfinite Decimal values.
3. **Market Data Protocols:** Structural quote and FX provider interfaces, with independent runtime checks for data validity and timestamp availability.
4. **Immutable Snapshot Engine:** Snapshot source metadata, all holdings, quotes, FX rates, and pricing conventions into independent immutable records.
5. **Valuation Engine:** Convert every native position to a common valuation currency with explicit unit semantics; reconcile position-level and portfolio-level values.
6. **Risk Gate:** Support gross and net exposure, single-name concentration, sector, currency, and liquidity constraints, with deterministic violation records.
7. **Type Checking:** Optional strict `mypy`/Pyright pipelines, positive fixtures, expected-negative fixtures, and cross-version annotation tests.
8. **Annotation Inspection:** AST structural audit, safe metadata retrieval, and documented rules for Python 3.10–3.14+.
9. **Numerical Verification:** Decimal reference cases, exact accounting identities, tolerance policies for externally quoted rates, and no premature rounding.
10. **Research Governance:** Explicit data provenance, versioning, immutable experiment configurations, reproducible runs, and audit reports.
11. **Testing and Failure Modes:** Type mismatch, stale quotes, missing FX, duplicate tickers, invalid prices, malformed provider results, broken static type contracts, and source mutation.
12. **Engineering Delivery:** Package layout, API docs, type-check configuration, unit tests, CI, executable tutorial notebook, and documented risk limitations.

### Acceptance Criteria

| Criterion | Weight |
|---|---:|
| Static contracts and generic design | 20% |
| Runtime input validation | 20% |
| Multi-currency numerical accuracy | 20% |
| Snapshot integrity and temporal correctness | 15% |
| Tests and failure analysis | 15% |
| Documentation and reproducibility | 10% |

### Final Goal

Produce a GitHub-ready quantitative research system that demonstrates mature understanding of Python typing **without overclaiming what annotations can prove**. Show independently that its financial data contracts are enforced, its monetary calculations reconcile, its historical inputs remain controlled, and its results can be reproduced under the declared Python version and market-data assumptions.